In [0]:
%pip install databricks-ai-search
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from databricks.ai_search.client import VectorSearchClient

In [0]:
dbutils.widgets.text("catalog_name", "rag_agentic")
dbutils.widgets.text("schema_name", "workday_demos")
dbutils.widgets.text("customer_table", "customer_feedback_kb")
dbutils.widgets.text("notes_table", "meeting_notes_kb")
dbutils.widgets.text("email_table", "email_communications_kb")
dbutils.widgets.text("customer_vs_index", "customer_feedback_index")
dbutils.widgets.text("notes_vs_index", "meeting_notes_index")
dbutils.widgets.text("email_vs_index", "email_communications_index")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
customer_table = dbutils.widgets.get("customer_table")
notes_table = dbutils.widgets.get("notes_table")
email_table = dbutils.widgets.get("email_table")
customer_vs_index = dbutils.widgets.get("customer_vs_index")
notes_vs_index = dbutils.widgets.get("notes_vs_index")
email_vs_index = dbutils.widgets.get("email_vs_index")

vs_endpoint_name = f"sales-endpoint-{catalog_name}"

feedback_vs_input_table = f"{catalog_name}.{schema_name}.{customer_table}"
notes_vs_input_table = f"{catalog_name}.{schema_name}.{notes_table}"
email_vs_input_table = f"{catalog_name}.{schema_name}.{email_table}"

feedback_vs_index_name = f"{catalog_name}.{schema_name}.{customer_vs_index}"
notes_vs_index_name = f"{catalog_name}.{schema_name}.{notes_vs_index}"
email_vs_index_name = f"{catalog_name}.{schema_name}.{email_vs_index}"

In [0]:
# Create vector search endpoint
client = VectorSearchClient(disable_notice=True)

try:
    client.get_endpoint(vs_endpoint_name)
    print(f"Vector search endpoint '{vs_endpoint_name}' exists")
except Exception as e:
    print(f"Vector search endpoint '{vs_endpoint_name}' did not exist")
    client.create_endpoint(
                            name=vs_endpoint_name,
                            endpoint_type="STANDARD"
                        )
    print(f"Vector search endpoint '{vs_endpoint_name}' created successfully")

ℹ️  Vector search endpoint 'sales-endpoint-rag_agentic' exists


In [0]:
def create_vs_index(endpoint_name, source_table, index_name):
    """Create a vector search index with error handling"""
    try:
        index = client.create_delta_sync_index(
                                                endpoint_name=endpoint_name,
                                                source_table_name=source_table,
                                                index_name=index_name,
                                                pipeline_type="TRIGGERED",
                                                primary_key="id",
                                                embedding_source_column="content",
                                                embedding_model_endpoint_name="databricks-bge-large-en"
                                            )
        print(f"{index_name} created successfully")
        return index
    
    except Exception as e:
        if "already exists" in str(e).lower():
            print(f"{index_name} already exists")

        else:
            print(f"Error creating {index_name}: {str(e)}")
            return None

# Create all three indexes
email_index = create_vs_index(vs_endpoint_name, email_vs_input_table, email_vs_index_name)

notes_index = create_vs_index(vs_endpoint_name, notes_vs_input_table, notes_vs_index_name)

feedback_index = create_vs_index(vs_endpoint_name, feedback_vs_input_table, feedback_vs_index_name)

✅ rag_agentic.workday_demos.email_communications_index created successfully
✅ rag_agentic.workday_demos.meeting_notes_index created successfully
✅ rag_agentic.workday_demos.customer_feedback_index created successfully


In [0]:
def sync_vs_index(endpoint_name, index_name):
    """Sync a vector search index to pick up new data from source table"""
    try:
        index = client.get_index(endpoint_name=endpoint_name, 
                                index_name=index_name)
        index.sync()
        print(f"{index_name} sync triggered")
        return index
    except Exception as e:
        print(f"Error syncing {index_name}: {str(e)}")
        return None

# Sync all three indexes to pick up new data
print("Syncing vector search indexes...")
sync_vs_index(vs_endpoint_name, email_vs_index_name)
sync_vs_index(vs_endpoint_name, notes_vs_index_name)
sync_vs_index(vs_endpoint_name, feedback_vs_index_name)
print("All indexes synced")

🔄 Syncing vector search indexes...
❌ Error syncing rag_agentic.workday_demos.email_communications_index: Vector index rag_agentic.workday_demos.email_communications_index is not ready.
❌ Error syncing rag_agentic.workday_demos.meeting_notes_index: Vector index rag_agentic.workday_demos.meeting_notes_index is not ready.
❌ Error syncing rag_agentic.workday_demos.customer_feedback_index: Vector index rag_agentic.workday_demos.customer_feedback_index is not ready.
✅ All indexes synced
